# AdaDetect on ChestMNIST
This notebook mirrors your MNIST workflow but uses ChestMNIST from MedMNIST.

In [1]:
# Run this once if medmnist is not installed:
# %pip install medmnist

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid

import medmnist
from medmnist import INFO

try:
    from .algo import EmpBH, adaptiveEmpBH
except ImportError:
    from algo import EmpBH, adaptiveEmpBH

np.random.seed(0)
torch.manual_seed(0)

KeyboardInterrupt: 

In [ ]:
def get_fdp(ytrue, rejection_set):
    if rejection_set.size:
        fdp = np.sum(ytrue[rejection_set] == 0) / len(rejection_set)
        tdp = np.sum(ytrue[rejection_set] == 1) / np.sum(ytrue == 1)
    else:
        fdp = 0
        tdp = 0
    return fdp, tdp

In [ ]:
data_flag = 'chestmnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

transform = transforms.ToTensor()
train_data = DataClass(split='train', transform=transform, download=True)

print('Task:', info['task'])
print('Labels:', info['label'])
print('Train size:', len(train_data))

In [ ]:
# ChestMNIST is multi-label, so we pick one pathology and build a binary null/alternative split.
target_name = 'cardiomegaly'
label_map = {int(k): v.lower() for k, v in info['label'].items()}
target_idx = [k for k, v in label_map.items() if v == target_name][0]

targets = torch.tensor(train_data.labels[:, target_idx]).squeeze()

inlr_labels = [0]
outlr_labels = [1]

inlr = (targets[..., None] == torch.tensor(inlr_labels)).any(-1).nonzero(as_tuple=True)[0]
outlr = (targets[..., None] == torch.tensor(outlr_labels)).any(-1).nonzero(as_tuple=True)[0]

print('Target pathology:', target_name, '(index', target_idx, ')')
print('Nulls:', len(inlr), 'Outliers:', len(outlr))

In [ ]:
class custom_subset(Dataset):
    def __init__(self, dataset, indices, labels=None):
        self.dataset = torch.utils.data.Subset(dataset, indices)
        all_labels = torch.tensor(dataset.labels[:, target_idx]).squeeze()
        self.targets = all_labels[indices] if labels is None else labels

    def __getitem__(self, idx):
        image = self.dataset[idx][0]
        target = self.targets[idx]
        return (image, target)

    def __len__(self):
        return len(self.targets)

In [ ]:
class BinaryConvNet(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.layer3 = nn.Sequential(
            nn.Conv2d(16, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.layer4 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.layer5 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [ ]:
class NNClassifier:
    def __init__(self, model, batch_size=64, n_epochs=10):
        self.model = model
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.loss_fn = nn.BCEWithLogitsLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)

    def fit(self, dataset):
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        for _ in range(self.n_epochs):
            for x, y in loader:
                self.model.train()
                y = y.float().unsqueeze(1)

                logits = self.model(x)
                loss = self.loss_fn(logits, y)

                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()

    def predict_proba(self, dataset):
        loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)

        with torch.no_grad():
            for x, _ in loader:
                self.model.eval()
                logits = self.model(x)
                probs = torch.sigmoid(logits)
                return probs.numpy().flatten()

In [ ]:
level = 0.1
test_size = 1000
null_prop = 0.9

n_inlr = int(test_size * null_prop)
n_outlr = test_size - n_inlr

calib_size = 1000
train_size = 3000
n_runs = 10

fdp, tdp = [0.] * n_runs, [0.] * n_runs

for i in range(n_runs):
    test_idx = np.concatenate([
        np.random.choice(inlr, n_inlr, replace=False),
        np.random.choice(outlr, n_outlr, replace=False)
    ])

    test_dataset = custom_subset(train_data, test_idx, torch.ones(test_size))

    train_idx = np.setdiff1d(inlr, test_idx)
    np.random.shuffle(train_idx)

    calib_idx = train_idx[:calib_size]
    train_idx2 = train_idx[calib_size:calib_size + train_size]

    calib_dataset = custom_subset(train_data, calib_idx, torch.ones(calib_size))
    train_dataset = custom_subset(train_data, train_idx2, torch.zeros(train_size))

    training_dataset = ConcatDataset([train_dataset, calib_dataset, test_dataset])

    n_channels = train_data[0][0].shape[0]
    model = BinaryConvNet(n_channels)
    clf = NNClassifier(model, batch_size=64, n_epochs=10)
    clf.fit(training_dataset)

    test_statistics = clf.predict_proba(test_dataset)
    null_statistics = clf.predict_proba(calib_dataset)

    rej_set = EmpBH(null_statistics, test_statistics, level=level)

    test_labels = np.concatenate([np.zeros(n_inlr), np.ones(n_outlr)])
    fdp[i], tdp[i] = get_fdp(test_labels, rej_set)

print('FDP:', np.mean(fdp), np.std(fdp))
print('TDP:', np.mean(tdp), np.std(tdp))

last_test_labels = test_labels
last_test_statistics = test_statistics
last_null_statistics = null_statistics

In [ ]:
print('Last run rejections:', len(rej_set), 'out of', len(test_dataset))

idx_rej = rej_set
idx_acc = np.setdiff1d(np.arange(len(test_dataset)), idx_rej)

if len(idx_rej) == 0:
    print('No rejections in last run.')
else:
    sample_rej = np.random.choice(idx_rej, size=min(10, len(idx_rej)), replace=False)
    sample_acc = np.random.choice(idx_acc, size=min(38, len(idx_acc)), replace=False)

    samples = np.concatenate([sample_rej, sample_acc])
    np.random.shuffle(samples)

    images = [test_dataset[i][0][0] for i in samples]

    fig = plt.figure(figsize=(6., 6.))
    grid = ImageGrid(fig, 111, nrows_ncols=(6, 4), axes_pad=0.05)

    for ax, idx, im in zip(grid, samples, images):
        ax.imshow(im, cmap='gray')
        ax.tick_params(bottom=False, top=False, left=False, right=False,
                      labelbottom=False, labeltop=False, labelleft=False, labelright=False)

        if idx in sample_rej:
            ax.spines['bottom'].set_color('crimson')
            ax.spines['top'].set_color('crimson')
            ax.spines['right'].set_color('crimson')
            ax.spines['left'].set_color('crimson')
            ax.spines['bottom'].set_linewidth(5)
            ax.spines['top'].set_linewidth(5)
            ax.spines['right'].set_linewidth(5)
            ax.spines['left'].set_linewidth(5)

    plt.show()